# SSO Coverage by Latitude Band

Computes 30-day coverage statistics for a 550 km sun-synchronous orbit
(LTAN 10:30, descending node) carrying a 10° half-angle nadir sensor.

Results are reported per 5° latitude band:
- Number of sample points in the band
- Cumulative coverage fraction (% of points seen ≥ once)
- Mean revisit time (hours)
- Maximum revisit time (hours)

**Object API** — uses `Spacecraft.sunsync`, `Sensor`, `AoI.from_region`, and `Coverage`.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from missiontools import Spacecraft, Sensor, AoI, Coverage

## 1. Spacecraft and Sensor

A nadir-pointing 10° half-angle sensor is body-mounted along the spacecraft
nadir axis (body-z = nadir, body-vector `[0, 0, 1]` in the sensor convention).

In [ ]:
EPOCH = np.datetime64('2025-01-01T00:00:00', 'us')

sc = Spacecraft.sunsync(
    altitude_km    = 550.0,
    node_solar_time = '10:30',
    node_type      = 'descending',
    epoch          = EPOCH,
)

sensor = Sensor(half_angle_deg=10.0, body_vector=[0, 0, 1])
sc.add_sensor(sensor)

period_s = 2 * np.pi * np.sqrt(sc.a**3 / sc.central_body_mu)
swath_km = 2 * (sc.a - sc.central_body_radius) * np.tan(np.radians(10.0)) / 1e3

print(f"Semi-major axis : {sc.a / 1e3:.1f} km")
print(f"Inclination     : {np.degrees(sc.i):.3f}°")
print(f"Orbital period  : {period_s / 60:.1f} min")
print(f"Propagator      : {sc.propagator_type}")
print(f"FOV half-angle  : {np.degrees(sensor.half_angle_rad):.0f}°")
print(f"Ground swath    : ~{swath_km:.0f} km")

## 2. Per-Band Coverage Analysis

For each 5° latitude band we create an `AoI.from_region`, attach a fresh
`Coverage` object, and compute coverage fraction and revisit time.

> **Note** — 36 bands × 30 days takes a few minutes.

In [ ]:
T_START = EPOCH
T_END   = EPOCH + np.timedelta64(30 * 86_400, 's')

MAX_STEP     = np.timedelta64(20, 's')
POINT_DENSITY = 200_000   # km²/point  (~450 km resolution)

LAT_EDGES = np.arange(-90, 91, 5)   # 37 edges → 36 bands

def band_label(lo_deg, hi_deg):
    lo_s = f"{abs(lo_deg):.0f}°{'S' if lo_deg <  0 else 'N'}"
    hi_s = f"{abs(hi_deg):.0f}°{'S' if hi_deg <= 0 else 'N'}"
    return f"{lo_s} – {hi_s}"

rows = []   # (label, n_pts, cov_pct, mean_rev_h, max_rev_h)

t0 = time.perf_counter()

for lo_deg, hi_deg in zip(LAT_EDGES[:-1], LAT_EDGES[1:]):
    aoi = AoI.from_region(
        lat_min_deg = float(lo_deg),
        lat_max_deg = float(hi_deg),
        point_density = POINT_DENSITY,
    )
    n = len(aoi)

    cov = Coverage(aoi, [sensor])

    cf = cov.coverage_fraction(T_START, T_END, max_step=MAX_STEP)
    rt = cov.revisit_time(T_START, T_END, max_step=MAX_STEP)

    cov_pct    = cf['final_cumulative'] * 100.0
    mean_rev_h = rt['global_mean'] / 3600.0 if not np.isnan(rt['global_mean']) else float('nan')
    max_rev_h  = rt['global_max']  / 3600.0 if not np.isnan(rt['global_max'])  else float('nan')

    rows.append((band_label(lo_deg, hi_deg), n, cov_pct, mean_rev_h, max_rev_h))

elapsed = time.perf_counter() - t0
print(f"Done in {elapsed:.1f} s")

## 3. Coverage Table

In [ ]:
print(f"  {'Band':>13}  {'Pts':>5}  {'Coverage':>10}  {'Mean Rev':>10}  {'Max Rev':>10}")
print(f"  {'─'*13}  {'─'*5}  {'─'*10}  {'─'*10}  {'─'*10}")

for label, n, cov_pct, mean_rev_h, max_rev_h in rows:
    mean_s = f"{mean_rev_h:8.2f} h" if not np.isnan(mean_rev_h) else "       —  "
    max_s  = f"{max_rev_h:8.2f} h" if not np.isnan(max_rev_h)  else "       —  "
    print(f"  {label:>13}  {n:>5}  {cov_pct:>9.1f}%  {mean_s}  {max_s}")

## 4. Visualisation

In [ ]:
labels    = [r[0] for r in rows]
lat_mids  = [(lo + hi) / 2 for lo, hi in zip(LAT_EDGES[:-1], LAT_EDGES[1:])]
cov_pcts  = np.array([r[2] for r in rows])
mean_revs = np.array([r[3] for r in rows])
max_revs  = np.array([r[4] for r in rows])

# Colour bars by coverage fraction
norm    = mcolors.Normalize(vmin=0, vmax=100)
cmap    = plt.cm.RdYlGn
colours = [cmap(norm(v)) for v in cov_pcts]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 10), sharey=True)

# --- Coverage fraction ---
ax1.barh(lat_mids, cov_pcts, height=4.5, color=colours, edgecolor='white', linewidth=0.4)
ax1.axvline(100, color='tab:green', linestyle='--', linewidth=1.0, alpha=0.7, label='100% coverage')
ax1.set_xlabel('Cumulative coverage fraction (%)')
ax1.set_ylabel('Latitude (°)')
ax1.set_title('30-day coverage fraction')
ax1.set_xlim(0, 105)
ax1.set_yticks(lat_mids[::2])
ax1.set_yticklabels([f"{int(l):+d}°" for l in lat_mids[::2]])
ax1.grid(True, axis='x', alpha=0.3)
ax1.legend(fontsize=9)

# Add value labels
for lat, v in zip(lat_mids, cov_pcts):
    if v > 0:
        ax1.text(min(v + 1, 103), lat, f"{v:.0f}%", va='center', fontsize=7)

# --- Mean revisit time ---
valid = ~np.isnan(mean_revs)
ax2.barh(np.array(lat_mids)[valid], mean_revs[valid], height=4.5,
         color='tab:blue', alpha=0.7, edgecolor='white', linewidth=0.4)
ax2.set_xlabel('Mean revisit time (hours)')
ax2.set_title('30-day mean revisit time')
ax2.grid(True, axis='x', alpha=0.3)

# Inclination limit annotation
ax2.axhline(np.degrees(sc.i) - 90, color='grey', linestyle=':', linewidth=1.0,
            label=f'Inclination limit ({np.degrees(sc.i):.1f}°)')
ax2.axhline(-(np.degrees(sc.i) - 90), color='grey', linestyle=':', linewidth=1.0)
ax2.legend(fontsize=9)

fig.suptitle(
    f'550 km SSO  |  LTAN 10:30  |  10° half-angle nadir sensor  |  30-day window',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()